# Arnold Schönberg – Universal-Edition: Topic Modeling v2 (2026)

Katharina Bleier · Aktualisierung von v1 (Machine Learning WS 2024/25, siehe `v1_2025/`)

**Ziel:** von der TEI-Edition über Topics zu inhaltlich qualifizierten Korrespondenzbeziehungen
(*wer mit wem* → *worüber*).

**Änderungen gegenüber v1**
| v1 (2025) | v2 (2026) |
|---|---|
| Daten: unveröffentlichte XML (Edition v1.1, 1.393 Briefe) | finaler, öffentlicher Datensatz (mdwRepository, CC BY-SA 4.0) |
| nur `body//p`, `note` ausgelassen | zusätzlich ohne `opener`/`closer`; `choice`, `del`/`add`, `app` gezielt; `lb break="no"` verbindet Wörter |
| keine Metadaten | Absender:in, Empfänger:in, Datum, Erwähnungen aus TEI |
| Kleinschreibung + Stoppwörter **vor** dem Embedding | Originalsätze ins Embedding, Stoppwörter nur für die Topic-Darstellung |
| Fragmente („1 u.", „ii.") als Dokumente | Einheiten unter 4 Wörtern gefiltert |
| kein Seed, Modell als pickle | Seed, Embedding-Cache, safetensors |
| nur Topic-Liste | Profile je Partner:in und Zeitraum, Netzwerk-Export (Gephi) |

**Daten:** ZIP „ASUE Data" aus dem mdwRepository herunterladen und die TEI-Dateien der Briefe nach `data/tei/` entpacken
(Registerdateien dürfen mit hinein – sie werden automatisch übersprungen).

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from asue_topics import extract, model, aggregate as ag

TEI_DIR = ROOT / "data" / "tei"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

## 1. Kodierung prüfen
Welche Elemente kommen im `<body>` vor? Mit den Kodierrichtlinien abgleichen und ggf. `skip_elements` anpassen
(z. B. falls Editionskommentare anders als mit `<note>` kodiert sind).

In [ ]:
inv = extract.element_inventory(TEI_DIR)
pd.Series(inv).sort_values(ascending=False).head(40)

## 2. Extraktion
Einheit: `"sentence"` (wie v1), alternativ `"paragraph"` oder `"letter"`.

In [ ]:
xcfg = extract.ExtractConfig(unit="sentence")
# Beispiel für Anpassungen:
# xcfg.skip_elements |= {"seg"}            # weiteres Element auslassen
# xcfg.skip_elements -= {"closer"}         # Briefschluss doch aufnehmen
letters_raw, units_raw = extract.run(TEI_DIR, OUT, xcfg)

In [ ]:
letters = pd.read_csv(OUT / "letters.csv")
units = pd.read_csv(OUT / "units.csv")
print(letters["year"].describe())
print(units["n_words"].describe())
units.sample(10, random_state=1)

**Stichprobe gegen die Edition prüfen:** einige Briefe in der Webansicht mit `units.csv` vergleichen
(Worttrennungen, Abkürzungen, gestrichene Stellen, Kommentare).

## 3. Topic Modeling mit BERTopic
Der erste Lauf berechnet die Embeddings (dauert; werden in `output/cache/` gespeichert).
Parameter, die das Ergebnis am stärksten beeinflussen: `min_cluster_size`, `min_samples`, `nr_topics`.

In [ ]:
mcfg = model.ModelConfig(
    min_words=4,
    embedding_model="paraphrase-multilingual-mpnet-base-v2",
    min_cluster_size=15,
    nr_topics=None,          # erst ohne Reduktion ansehen, dann z. B. 30
    reduce_outliers=False,
)
units_f = model.load_units(OUT, mcfg)

In [ ]:
topic_model, res_bt, info_bt, emb = model.fit_bertopic(units_f, mcfg, OUT / "cache")
model.export(res_bt, info_bt, OUT, "bertopic")
info_bt.head(30)

In [ ]:
model.save_bertopic(topic_model, OUT / "bertopic_model", mcfg)
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart(top_n_topics=12, n_words=8)

### Topics benennen
`output/bertopic_topic_labels.csv` öffnen und die Spalte **label** ausfüllen (z. B. „Vertrag / Rechte",
„Korrektur Aufführungsmaterial"). Die Datei wird bei erneutem Lauf nicht überschrieben –
**nach einer Änderung von Parametern die Datei löschen**, da sich die Topic-Nummern ändern.

## 4. Vergleich: NMF (ohne Sprachmodell)
Gleiche Daten, anderes Verfahren – zeigt, wie stark Topics vom Modell abhängen.

In [ ]:
_, res_nmf, info_nmf = model.fit_nmf(units_f, mcfg, n_topics=20)
model.export(res_nmf, info_nmf, OUT, "nmf")
info_nmf

## 5. Von Sätzen zu Beziehungen
Satz-Topics → Topic-Anteile pro Brief → Profile pro Korrespondenzpartner:in und Zeitraum.

In [ ]:
PREFIX = "bertopic"          # oder "nmf"
res = res_bt if PREFIX == "bertopic" else res_nmf
labels = model.load_labels(OUT, PREFIX)

letters = ag.add_period(ag.add_partner(pd.read_csv(OUT / "letters.csv"), xcfg.schoenberg_name_pattern))
lt = ag.letter_topics(res, include_outliers=False)
letters[["direction", "partner"]].value_counts().head(20)

In [ ]:
prof_partner = ag.profile(lt, letters, "partner", min_letters=15)
prof_period = ag.profile(lt, letters, "period")
pd.concat([prof_partner["n_letters"], ag.topic_diversity(prof_partner)], axis=1)

In [ ]:
ag.plot_heatmap(prof_partner, labels, OUT / f"{PREFIX}_heatmap_partner.png",
                "Topic-Profile der Korrespondenzbeziehungen")
ag.plot_heatmap(prof_period, labels, OUT / f"{PREFIX}_heatmap_period.png",
                "Topic-Profile nach Zeitraum")
ag.plot_topics_over_time(lt, letters, labels, OUT / f"{PREFIX}_topics_time.png", top_n=6)

G_pt = ag.partner_topic_graph(prof_partner, labels, min_share=0.12)
ag.plot_partner_topic_graph(G_pt, OUT / f"{PREFIX}_partner_topic_net.png",
                            "Korrespondenzpartner:innen und ihre Themen")
ag.write_gexf(G_pt, OUT / f"{PREFIX}_partner_topic.gexf")
ag.write_gexf(ag.correspondence_graph(lt, letters, labels), OUT / f"{PREFIX}_correspondence.gexf")

## 6. Close Reading
Für ein Topic und eine Beziehung die typischsten Sätze ansehen – trägt die Zuordnung?

In [ ]:
TOPIC, PARTNER = 0, prof_partner.index[0]
ids = letters.loc[letters["partner"] == PARTNER, "letter_id"]
sel = res[(res["topic"] == TOPIC) & (res["letter_id"].isin(ids))]
print(labels.get(TOPIC), "|", PARTNER, "|", len(sel), "Sätze")
sel[["letter_id", "text"]].head(15)

## 7. Dokumentation für den Workshop
Festhalten: Datensatz-Version, Zahl der Briefe/Sätze, Parameter (`mcfg`), Zahl der Topics, Outlier-Anteil,
Vergleich mit v1 (231 Topics, 44 % Outlier bei 15.328 Sätzen).

In [ ]:
print("Briefe:", len(letters), "| Einheiten modelliert:", len(units_f))
print("Topics:", (info_bt.topic != -1).sum(), "| Outlier:", f"{(res_bt.topic == -1).mean():.0%}")
print(mcfg)